<a href="https://colab.research.google.com/github/look4pritam/NetflixPrize/blob/master/Notebooks/v1.0.0/v1.0.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Netflix Prize

## Install Python packages.

In [ ]:
!pip install numpy pandas scipy tqdm

In [ ]:
!pip install implicit

## Set the project root directory.

In [ ]:
import os

root_dir = '/content/'
os.chdir(root_dir)

!ls -al

## Download the Netflix Prize dataset from Kaggle.

### Set the Kaggle API key.

### Generate and upload 'kaggle.json' file.

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = root_dir
!chmod 600 /content/kaggle.json

### Download the Netflix Prize dataset.

In [ ]:
!kaggle datasets download netflix-inc/netflix-prize-data
!ls -al

In [ ]:
import os
import zipfile

### Extract the Netflix Prize dataset.

In [ ]:
zip_path = "netflix-prize-data.zip"
dataset_path = "netflix_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

print("Extracted Netflix Prize dataset from:", os.listdir(dataset_path))

## Load the Netflix Prize dataset.

### Import Python packages.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
def build_sparse_matrix(input_folder):

    user_ids = []
    movie_ids = []
    ratings = []

    user_map = {}
    movie_map = {}

    user_counter = 0
    movie_counter = 0

    files = [
        "combined_data_1.txt",
        "combined_data_2.txt",
        "combined_data_3.txt",
        "combined_data_4.txt"
    ]

    for file in files:

        path = os.path.join(input_folder, file)

        print("\nProcessing input file: ", file)

        movie_id = None

        with open(path, "r", encoding="latin-1") as f:

            for line in tqdm(f):

                line = line.strip()

                if not line:
                    continue

                # movie ID line
                if line.endswith(":"):

                    movie = int(line[:-1])

                    if movie not in movie_map:
                        movie_map[movie] = movie_counter
                        movie_counter += 1

                    movie_id = movie_map[movie]

                else:

                    parts = line.split(",")

                    if len(parts) < 2:
                        continue

                    user = int(parts[0])
                    rating = float(parts[1])

                    if user not in user_map:
                        user_map[user] = user_counter
                        user_counter += 1

                    user_ids.append(user_map[user])
                    movie_ids.append(movie_id)
                    ratings.append(rating)

    rating_matrix = coo_matrix(
        (ratings, (user_ids, movie_ids)),
        shape=(user_counter, movie_counter)
    )

    return rating_matrix.tocsr(), user_map, movie_map

In [ ]:
rating_matrix, user_map, movie_map = build_sparse_matrix("netflix_dataset")

print("Matrix shape: ", rating_matrix.shape)
print("Number of ratings: ", rating_matrix.nnz)

## Factorize matrix using Alternating Least Squares (ALS).

In [ ]:
def matrix_factorization(rating_matrix, embedding_size=20, iterations=5, regularization_factor=0.1):

    user_count, sample_count = rating_matrix.shape

    user_features = np.random.normal(scale=1./embedding_size, size=(user_count, embedding_size))
    movie_features = np.random.normal(scale=1./embedding_size, size=(sample_count, embedding_size))

    rows = rating_matrix.row
    cols = rating_matrix.col
    ratings = rating_matrix.data

    for current_iteration in range(iterations):

        print("Iteration: ", current_iteration+1)

        # Update Users
        for u in tqdm(range(user_count)):

            idx = rows == u
            items = cols[idx]
            r = ratings[idx]

            if len(items) == 0:
                continue

            V_i = movie_features[items]

            A = V_i.T @ V_i + regularization_factor*np.eye(embedding_size)
            b = V_i.T @ r

            user_features[u] = np.linalg.solve(A, b)

        # Update Movies
        for i in tqdm(range(sample_count)):

            idx = cols == i
            users = rows[idx]
            r = ratings[idx]

            if len(users) == 0:
                continue

            U_u = user_features[users]

            A = U_u.T @ U_u + regularization_factor*np.eye(embedding_size)
            b = U_u.T @ r

            movie_features[i] = np.linalg.solve(A, b)

    return user_features, movie_features

In [ ]:
user_features, movie_features = matrix_factorization(rating_matrix, embedding_size=20, iterations=5)

## Compute the predicted matrix.

In [ ]:
predicted_matrix = user_features @ movie_features.T

## Computer the RMSE metric.

In [ ]:
def compute_rmse(rating_matrix, predicted_matrix):

    rows = rating_matrix.row
    cols = rating_matrix.col
    ground_truth_ratings = rating_matrix.data

    predicted_ratings = predicted_matrix[rows, cols]

    rmse = np.sqrt(np.mean((ground_truth_ratings - predicted_ratings)**2))

    return rmse

In [ ]:
rmse = compute_rmse(rating_matrix, predicted_matrix)
print("RMSE: ", rmse)